In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%cudf.pandas.profile
### cell 10 ###

### cell 10 (optimized for cuDF)

def combine_subset_of_data_from_multiple_years_22(question, x_axis_title, include_2017=None):
    # compute percent distribution in one vectorized step
    def count_percent(series):
        return (series.value_counts(dropna=False, normalize=True) * 100).round(1)

    # mappings for gender question
    gender_map = {
        'Male': 'Man',
        'Female': 'Woman',
        'Nonbinary': 'Prefer to self-describe',
        'Prefer not to say': 'Prefer to self-describe'
    }
    gender_map_2017 = gender_map.copy()
    gender_map_2017.update({
        'A different identity': 'Prefer to self-describe',
        'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe'
    })

    # map each year to its dataframe
    df_map = {
        2022: responses_df_2022_cell10,
        2021: responses_df_2021,
        2020: responses_df_2020,
        2019: responses_df_2019_cell10,
        2018: responses_df_2018_cell10,
        2017: responses_df_2017
    }

    years = [2022, 2021, 2020, 2019, 2018]
    if include_2017:
        years.append(2017)

    parts = []
    for yr in years:
        df = df_map[yr]
        # for 2017 the column is still 'GenderSelect', otherwise use the question
        col_in = 'GenderSelect' if yr == 2017 else question
        series = df[col_in]
        # apply gender normalization only when this is the gender question
        if question == 'What is your gender? - Selected Choice':
            series = series.replace(gender_map_2017 if yr == 2017 else gender_map)
        # get percentages
        pct = count_percent(series)
        # turn into a tidy dataframe and add year
        df_year = pct.rename_axis(x_axis_title).reset_index(name='percentage')
        df_year['year'] = str(yr)
        parts.append(df_year[['percentage', 'year', x_axis_title]])

    # concatenate and sort
    df_combined = pd.concat(parts, ignore_index=True)
    df_combined = df_combined.sort_values(by=['year', 'percentage'], ascending=True)
    return df_combined

# define variables if not already defined
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''

# call the optimized function
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22,
    title_for_x_axis_cell22,
    include_2017=True
)
# ensure correct sorting and view info
age_df_combined_cell22 = age_df_combined_cell22.sort_values(by=['year', 'percentage'], ascending=True)
age_df_combined_cell22.info()